# 02 Binary Logit: Utility Difference and Choice Probability

## Program Overview

This teaching program introduces binary Logit using a small commute mode-choice data set. It connects the Part 2 discussion of utility, random utility, probability, and likelihood to executable Python code.

## Overall Task AI Prompt (English)

Use this notebook as a programming lesson for discrete choice modeling. Before
running each code cell, read the prompt and explain what theoretical concept the
code implements. After running the cell, use the interpretation cell to connect
the numerical result back to utility, probability, likelihood, estimation, or
model validation.

## Learning Objectives

- Define the decision maker, alternatives, choice variable, and explanatory attributes.
- Translate random utility theory into executable Python code.
- Interpret model probabilities, coefficients, likelihood values, and validation results.
- Distinguish behavioral explanation from prediction accuracy.


## AI Prompt for This Code Cell

Create a small stated-choice style data set for a two-mode commute choice. Identify the chosen alternative and the time/cost attributes.

In [5]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    "person_id": np.arange(1, 13),
    "car_time": [25, 30, 18, 45, 28, 36, 22, 40, 33, 27, 31, 24],
    "transit_time": [35, 28, 24, 50, 34, 30, 32, 44, 29, 38, 36, 33],
    "car_cost": [6.0, 7.5, 5.0, 9.0, 6.5, 8.5, 5.5, 9.5, 7.0, 6.0, 7.2, 5.8],
    "transit_cost": [3.0, 3.5, 2.8, 4.5, 3.2, 3.8, 3.0, 4.2, 3.6, 3.1, 3.4, 3.0],
    "choice_car": [1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1],
})

data["time_diff"] = data["car_time"] - data["transit_time"]
data["cost_diff"] = data["car_cost"] - data["transit_cost"]
data

,person_id,car_time,transit_time,car_cost,transit_cost,choice_car,time_diff,cost_diff
0,1,25,35,6.0,3.0,1,-10,3.0
1,2,30,28,7.5,3.5,0,2,4.0
2,3,18,24,5.0,2.8,1,-6,2.2
3,4,45,50,9.0,4.5,0,-5,4.5
4,5,28,34,6.5,3.2,1,-6,3.3
5,6,36,30,8.5,3.8,0,6,4.7
6,7,22,32,5.5,3.0,1,-10,2.5
7,8,40,44,9.5,4.2,0,-4,5.3
8,9,33,29,7.0,3.6,0,4,3.4
9,10,27,38,6.0,3.1,1,-11,2.9


## Output Interpretation

This output defines a balanced binary choice teaching data set: 6 travelers chose car and 6 chose transit. The derived variable `time_diff = car_time - transit_time` ranges from -11 to 6 minutes, so some travelers face a faster car trip while others face a slower car trip. The cost difference is always positive, ranging from 2.2 to 5.3, meaning car is always more expensive than transit in this example. This matters for interpretation: because car has a cost disadvantage for every observation, the model must explain car choices mainly through time savings and the intercept. Students should read this table as the empirical setup for a utility-difference model, not just as raw data.


## AI Prompt for This Code Cell

Define a binary Logit probability function from the utility difference between car and transit.

In [6]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def car_choice_probability(beta_time, beta_cost, intercept=0.0):
    utility_difference = (
        intercept
        + beta_time * data["time_diff"]
        + beta_cost * data["cost_diff"]
    )
    return sigmoid(utility_difference)

data["p_car_example"] = car_choice_probability(
    beta_time=-0.10,
    beta_cost=-0.55,
    intercept=0.40,
)

data[["person_id", "time_diff", "cost_diff", "choice_car", "p_car_example"]]

,person_id,time_diff,cost_diff,choice_car,p_car_example
0,1,-10,3.0,1,0.437823
1,2,2,4.0,0,0.119203
2,3,-6,2.2,1,0.447692
3,4,-5,4.5,0,0.171505
4,5,-6,3.3,1,0.306826
5,6,6,4.7,0,0.058140
6,7,-10,2.5,1,0.506250
7,8,-4,5.3,0,0.107647
8,9,4,3.4,0,0.133542
9,10,-11,2.9,1,0.476268


## Output Interpretation

The calculated `p_car_example` values range from 0.058 to 0.506. This tells us that the proposed coefficients generally make car less likely than transit, which is reasonable because car is always more expensive in the data. The average predicted probability for observations where car was actually chosen is 0.436, while the average for observations where transit was chosen is only 0.137. That is a useful model signal: even though the probabilities are not perfect, the model assigns substantially higher car probability to actual car choosers. The negative `beta_time = -0.10` and `beta_cost = -0.55` mean that extra car time and extra car cost both reduce the utility of car relative to transit.


## AI Prompt for This Code Cell

Compute the log-likelihood for the proposed coefficients. Explain why a higher log-likelihood means the observed choices are better explained by the model.

In [7]:
def binary_log_likelihood(probabilities, choices):
    eps = 1e-12
    p = np.asarray(probabilities, dtype=float)
    y = np.asarray(choices, dtype=float)
    p = np.clip(p, eps, 1 - eps)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

ll_example = binary_log_likelihood(data["p_car_example"], data["choice_car"])
print(f"Log-likelihood of the example binary Logit model: {ll_example:.3f}")

Log-likelihood of the example binary Logit model: -5.952


## Output Interpretation

The example model has a log-likelihood of -5.952. Interpret this as the total log probability that the model assigns to the 12 observed choices. A less negative value would mean better fit. For comparison, if a model assigned very low probability to the choices that actually occurred, the log-likelihood would be much more negative. Here, the value is plausible for a small teaching example because the model separates car and transit choosers moderately well: actual car choosers receive an average car probability of 0.436, while transit choosers receive an average car probability of 0.137. This is the numerical version of the W3 MLE idea: good parameters are those that put probability mass on what people actually chose.


## AI Prompt for This Code Cell

Estimate the binary Logit coefficients by maximizing the log-likelihood with a small grid search. This mirrors the MLE logic introduced in the PPT without requiring specialized software.

In [8]:
best = {"log_likelihood": -np.inf}

for intercept in np.linspace(-1.2, 1.2, 25):
    for beta_time in np.linspace(-0.25, 0.02, 28):
        for beta_cost in np.linspace(-1.00, 0.02, 26):
            p_car = car_choice_probability(beta_time, beta_cost, intercept)
            ll = binary_log_likelihood(p_car, data["choice_car"])
            if ll > best["log_likelihood"]:
                best = {
                    "intercept": intercept,
                    "beta_time": beta_time,
                    "beta_cost": beta_cost,
                    "log_likelihood": ll,
                }

best

{'intercept': 1.2,
 'beta_time': -0.25,
 'beta_cost': -0.7552,
 'log_likelihood': -2.844379299140547}

## Output Interpretation

The grid search finds `intercept = 1.2`, `beta_time = -0.25`, `beta_cost = -0.7552`, with a best log-likelihood of -2.844. This improves substantially from the earlier example log-likelihood of -5.952, so the fitted parameters explain the observed choices better. The positive intercept means that, after accounting for time and cost differences, the fitted model needs a baseline preference for car to explain why 6 of 12 travelers still chose car even though car is always more expensive. The negative time coefficient says each extra minute of car disadvantage sharply lowers car utility; the negative cost coefficient says each extra unit of car cost disadvantage also lowers car utility. Notice that `beta_time = -0.25` is at the lower edge of the teaching grid, so students should not overinterpret it as a precise optimum. It suggests the grid should be widened or a numerical optimizer should be used in a real estimation exercise.
